In [13]:
import numpy as np
import pandas as pd

#任务1
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: f'{x:.2f}')
#1.输出
orders = pd.DataFrame({
    'order_id': [f'O{num}' for num in range(1001, 1019)],
    'region': ['华东','华北','华南','华东','西南','华北','华南','华东','西南','华北','华东','华南','西南','华东','华北','华南','华东','西南'],
    'product': ['机械键盘','无线鼠标','显示器','扩展坞','机械键盘','显示器','无线鼠标','显示器','扩展坞','机械键盘','显示器','扩展坞','显示器','机械键盘','扩展坞','显示器','无线鼠标','机械键盘'],
    'category': ['外设','外设','显示设备','配件','外设','显示设备','外设','显示设备','配件','外设','显示设备','配件','显示设备','外设','配件','显示设备','外设','外设'],
    'quantity': [2,3,1,4,5,2,6,1,3,2,8,2,1,3,5,2,4,6],
    'unit_price': [289,129,1299,399,289,1299,129,1299,399,289,1299,399,1299,289,399,1299,129,289],
    'member_level': ['金卡','普通','银卡','金卡','银卡','普通','金卡','银卡','普通','金卡','银卡','金卡','普通','银卡','金卡','金卡','普通','银卡'],
    'coupon_rate': [0.05,0.00,0.08,0.10,0.05,0.00,0.12,0.05,0.00,0.08,0.10,0.05,0.00,0.12,0.05,0.08,0.00,0.10],
    'salesperson': ['小林','小周','小陈','小林','小赵','小周','小陈','小林','小赵','小周','小林','小陈','小赵','小林','小周','小陈','小林','小赵']
})
orders
row, col = orders.shape
print(f"数据行数：{row}，列数：{col}")
print("全部列名：", orders.columns.tolist())
#2.取出列和类型
s_region = orders['region']
df_three = orders[['order_id','product','quantity']]
print("region单列类型：", type(s_region))
print("三列数据集类型：", type(df_three))
#3、4.用iloc，loc'查找
df_iloc = orders.iloc[3:8, 0:4]
print(df_iloc)
df_east = orders.loc[orders['region'] == '华东', ['order_id','product','member_level']]
print(df_east)
print("loc基于行列标签定位数据，且增删列是不会像iloc一样取错数字，维护成本更低")

数据行数：18，列数：9
全部列名： ['order_id', 'region', 'product', 'category', 'quantity', 'unit_price', 'member_level', 'coupon_rate', 'salesperson']
region单列类型： <class 'pandas.core.series.Series'>
三列数据集类型： <class 'pandas.core.frame.DataFrame'>
  order_id region product category
3    O1004     华东     扩展坞       配件
4    O1005     西南    机械键盘       外设
5    O1006     华北     显示器     显示设备
6    O1007     华南    无线鼠标       外设
7    O1008     华东     显示器     显示设备
   order_id product member_level
0     O1001    机械键盘           金卡
3     O1004     扩展坞           金卡
7     O1008     显示器           银卡
10    O1011     显示器           银卡
13    O1014    机械键盘           银卡
16    O1017    无线鼠标           普通
loc基于行列标签定位数据，且增删列是不会像iloc一样取错数字，维护成本更低


In [12]:
analysis = orders.assign(
    gross_amount = lambda x: (x['quantity'] * x['unit_price']).round(2),
    member_discount = lambda x: np.where(
        x['member_level'] == '金卡', 0.10,
        np.where(x['member_level'] == '银卡', 0.05, 0.00)
    ),
    payable_amount = lambda x: (
        x['gross_amount'] * (1 - x['member_discount']) * (1 - x['coupon_rate'])
    ).round(2),
    shipping_fee = lambda x: np.where(x['payable_amount'] >= 1000, 0, 20),
    final_amount = lambda x: (x['payable_amount'] + x['shipping_fee']).round(2)
)
show_cols = ['order_id','quantity','unit_price','gross_amount','member_discount','payable_amount','shipping_fee','final_amount']
print(analysis[show_cols].head(8))

  order_id  quantity  unit_price  gross_amount  member_discount  \
0    O1001         2         289           578             0.10   
1    O1002         3         129           387             0.00   
2    O1003         1        1299          1299             0.05   
3    O1004         4         399          1596             0.10   
4    O1005         5         289          1445             0.05   
5    O1006         2        1299          2598             0.00   
6    O1007         6         129           774             0.10   
7    O1008         1        1299          1299             0.05   

   payable_amount  shipping_fee  final_amount  
0          494.19            20        514.19  
1          387.00            20        407.00  
2         1135.33             0       1135.33  
3         1292.76             0       1292.76  
4         1304.11             0       1304.11  
5         2598.00             0       2598.00  
6          613.01            20        633.01  
7         11

In [11]:
cond1 = analysis['region'].isin(['华东','华南'])
cond2 = analysis['final_amount'] >= 700
cond3 = (analysis['quantity'] >= 2) | (analysis['member_level'] == '金卡')
mask = cond1 & cond2 & cond3
#括号内运算优先度大于><符号，能够保证运算顺序不出错
target_cols = ['order_id','region','product','quantity','member_level','final_amount']
focus_orders = analysis.loc[mask, target_cols].sort_values('final_amount', ascending=False)
print(focus_orders)

   order_id region product  quantity member_level  final_amount
10    O1011     华东     显示器         8           银卡       8885.16
15    O1016     华南     显示器         2           金卡       2151.14
3     O1004     华东     扩展坞         4           金卡       1292.76
13    O1014     华东    机械键盘         3           银卡        744.81
11    O1012     华南     扩展坞         2           金卡        702.29


In [10]:
def add_order_level(df):
    df_copy = df.copy()
    df_copy['order_level'] = np.where(
        df_copy['final_amount'] >= 2000, "战略订单",
        np.where(df_copy['final_amount'] >= 1000, "重点订单", "普通订单")
    )
    return df_copy
    
leveled_orders = analysis.pipe(add_order_level)
level_count = leveled_orders['order_level'].value_counts()
print("各等级订单数量：\n", level_count)

各等级订单数量：
 order_level
重点订单    8
普通订单    7
战略订单    3
Name: count, dtype: int64


In [9]:
region_report = (
    analysis
    .pipe(add_order_level)                         # 1. 添加订单等级
    .query("final_amount >= 500")                  # 2. 过滤实付金额500以上订单
    .groupby(['region', 'order_level'])            # 3. 按地区+订单等级分组
    .agg(
        order_count = ('order_id', 'count'),
        total_quantity = ('quantity', 'sum'),
        total_revenue = ('final_amount', 'sum'),
        avg_revenue = ('final_amount', 'mean')
    )                                              # 4. 聚合计算指标
    .sort_values('total_revenue', ascending=False) # 5. 按总营收降序
).round(2)
print(region_report)

                    order_count  total_quantity  total_revenue  avg_revenue
region order_level                                                         
华东     战略订单                   1               8        8885.16      8885.16
西南     重点订单                   4              15        5282.68      1320.67
华北     战略订单                   1               2        2598.00      2598.00
华东     重点订单                   2               5        2465.11      1232.55
华南     战略订单                   1               2        2151.14      2151.14
华东     普通订单                   3               9        1795.00       598.33
华北     重点订单                   1               5        1705.72      1705.72
华南     普通订单                   2               8        1335.30       667.65
       重点订单                   1               1        1135.33      1135.33


In [8]:
sales_total = analysis.groupby('salesperson')['final_amount'].sum().sort_values(ascending=False)
top_sales = sales_total.index[0]
top_sales_total = sales_total.iloc[0]
print(f"最终成交金额销售人员：{top_sales}，最终成交金额：{top_sales_total:.2f}")

sales_region = analysis.loc[analysis['salesperson'] == top_sales].groupby('region')['final_amount'].sum()
core_region = sales_region.idxmax()
core_region_amt = sales_region.max()
print(f"该销售员成交金额最高地区：{core_region}，地区成交金额：{core_region_amt:.2f}")

contribute_rate = (core_region_amt / top_sales_total) * 100
print(f"核心地区成交占该销售员总业绩比例：{contribute_rate:.2f}%")

print("\n业务结论：该销售员只在核心地区成交金额。")

最终成交金额销售人员：小林，最终成交金额：13145.27
该销售员成交金额最高地区：华东，地区成交金额：13145.27
核心地区成交占该销售员总业绩比例：100.00%

业务结论：该销售员只在核心地区成交金额。
